## Determining the optimal number of hidden layers and neurons for an Artificial Neural Networks (ANN)

This can be challenging and often rewuires experimentation. However, there are some guidelines and methods tat can help you in making an informed decision:

* **Start Simple:** Begin with a simple architecture and gradually increase complexity if needed.
* **Grid Search/Random Search:** Use grid search or random search to try different architectures.
* **Cross-Validation:** Use cross-validation to evaluate the performance of different architectures.
* **Heuristics and Rules of Thumb:** Some heuristics and empirical rules can provide starting points, such as: 
    * The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.
    * A common practice is to start with 1-2 hidden layers

In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import  KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [2]:
## Load the dataset
data=pd.read_csv("../Churn_Modelling.csv")


In [3]:
## Preprocess the data
### Drop irrelevant columns
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [4]:
## Encode categorical variables (gender)
label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [5]:
## OneHotEncoder for geography column

ohe=OneHotEncoder()
geo_encoder=ohe.fit_transform(data[['Geography']])
geo_encoder


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [6]:
encoded_columns=ohe.get_feature_names_out(['Geography'])
encoded_columns

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [7]:
## lets convert the encoded columns into pandas dataframe

geo_encoded_df=pd.DataFrame(geo_encoder.toarray(),columns=encoded_columns)

In [8]:
## combine columns with the original data
data=pd.concat([data.drop('Geography', axis=1),geo_encoded_df],axis=1)

data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [10]:
## Divide the dataset into dependent and independent
X=data.drop("Exited",axis=1)
y=data["Exited"]

In [11]:
## Split the data to train and test
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [12]:
## scale the features
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [15]:
## Save the encoders and scalar
with open("label_encoder_gender.pkl",'wb') as file:
    pickle.dump(label_encoder_gender,file)

with open("onehot_encoder_geo.pkl", "wb") as file:
    pickle.dump(ohe,file)

with open("scaler.pkl","wb") as file:
    pickle.dump(scaler,file)

In [23]:
## define a function to create a model and try different parameters (Keras Classifier)

def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_test_scaled.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss='binary_crossentropy', metrics=['accuracy'])

    return model

In [24]:
## create Keras classifier( this will create the entuire neural network). 
model=KerasClassifier(build_fn=create_model,neurons=32,layers=1,epochs=50,batch_size=10,verbose=0)

In [25]:
## Define the grid search parameters
param_grid={
    'neurons':[16,32,64,128],
    'layers':[1,2],
    # 'batch_size':[10,20], ## we will just stay with the 10 we assigned before
    'epochs':[50,100]
}

In [26]:
## Perform grid search
grid=GridSearchCV(estimator=model,param_grid=param_grid,n_jobs=1,cv=3)
grid_result=grid.fit(X_train_scaled,y_train)

## Print the best parameters
print("Best: %f using %s" % (grid_result.best_score_,grid_result.best_params_))

c:\Users\pc\Downloads\python_deep_learning\venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\pc\Downloads\python_deep_learning\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


c:\Users\pc\Downloads\python_deep_learning\venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\pc\Downloads\python_deep_learning\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\pc\Downloads\python_deep_learning\venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\pc\Downloads\python_deep_learning\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning:

Best: 0.858249 using {'epochs': 100, 'layers': 1, 'neurons': 32}
